In [5]:
import numpy as np
from pathlib import Path
import pickle

In [6]:
def effective_rank(A, eps=1e-12):
    """
    Compute the effective rank of a matrix A.

    Parameters
    ----------
    A : np.ndarray
        Input matrix.
    eps : float
        Small constant to avoid log(0).

    Returns
    -------
    erank : float
        Effective rank of A.
    singular_values : np.ndarray
        Singular values of A.
    """

    # Singular values
    singular_values = np.linalg.svd(A, compute_uv=False)

    # Normalize singular values into a probability distribution
    p = singular_values / (np.sum(singular_values) + eps)

    # Shannon entropy
    entropy = -np.sum(p * np.log(p + eps))

    # Effective rank
    erank = np.exp(entropy)

    return erank, singular_values

In [7]:
# N_LAYERS = [3, 5]
N_LAYERS = [3, 5, 7, 9, 11, 13, 15, 17, 20]
ROLL_LOC = Path("Results/rollout_local")
ROLL_GLOB = Path("Results/rollout_global")

attn_rank_dict = {}
roll_rank_dict = {}

for layer in N_LAYERS:

    attn_ranks = []
    
    with open(ROLL_LOC / f"{layer}layers/attn_matrices{layer}layers.pkl", "rb") as f:
        attn_matrices = pickle.load(f)
    
    attn_list = [np.mean(attn_matrices[key]['attn_matrix'], axis=0) 
                 for key in attn_matrices.keys()]

    for matrix in attn_list:
        erank, _ = effective_rank(matrix)

        attn_ranks.append(erank)

    attn_rank_dict[f"{layer}layers"] = attn_ranks     

    rollouts = np.load(ROLL_GLOB / f"{layer}layers/rollout{layer}layers.npz")
    R = rollouts['rollout']
    erank_R, _ = effective_rank(R)
    roll_rank_dict[f"{layer}layers"] = erank_R

# with open("Results/effective_ranks/layerwise_attention_ranks.pkl", "wb") as f:
#     pickle.dump(attn_rank_dict, f)

# with open("Results/effective_ranks/rollout_ranks.pkl", "wb") as f:
#     pickle.dump(roll_rank_dict, f)

In [8]:
def column_importance_count(A, percentile=90):
    """
    Count how many columns have column sum above a percentile threshold.

    Parameters
    ----------
    A : np.ndarray
        Attention or rollout matrix.
    percentile : float
        Percentile used as threshold, e.g. 90.

    Returns
    -------
    count : int
        Number of columns above threshold.
    column_sums : np.ndarray
        Sum of each column.
    threshold : float
        Percentile threshold.
    important_columns : np.ndarray
        Indices of columns above threshold.
    """

    column_sums = A.sum(axis=0)

    threshold = np.percentile(column_sums, percentile)

    important_columns = np.where(column_sums >= threshold)[0]

    count = len(important_columns)

    return count, important_columns

In [9]:
def column_importance_relative(A, alpha=0.5):
    column_sums = A.sum(axis=0)

    threshold = alpha * np.max(column_sums)

    important_columns = np.where(column_sums >= threshold)[0]

    count = len(important_columns)

    return count, important_columns

In [15]:
attn_count_dict = {}
roll_count_dict = {}
attn_count_dict_alpha = {}
roll_count_dict_alpha = {}

ALPHA = 0.9
ALPHA_s = f"{ALPHA:.2f}".replace(".", "")

for layer in N_LAYERS:

    attn_count = []
    attn_columns = []
    attn_count_rel = []
    attn_columns_rel = []
    
    with open(ROLL_LOC / f"{layer}layers/attn_matrices{layer}layers.pkl", "rb") as f:
        attn_matrices = pickle.load(f)
      
    attn_list = [np.mean(attn_matrices[key]['attn_matrix'], axis=0) 
                 for key in attn_matrices.keys()]

    for i, matrix in enumerate(attn_list):

        count, important_columns = column_importance_count(matrix, percentile=90)

        count_rel, important_columns_rel = column_importance_relative(matrix, alpha=ALPHA)

        attn_count.append(count)
        attn_columns.append(important_columns)
        attn_count_rel.append(count_rel)
        attn_columns_rel.append(important_columns_rel)

    attn_count_dict[f"{layer}layers"] = {"count": attn_count,
                                         "columns": attn_columns}

    attn_count_dict_alpha[f"{layer}layers"] = {"count": attn_count_rel, 
                                               "columns": attn_columns_rel}

    rollouts = np.load(ROLL_GLOB / f"{layer}layers/rollout{layer}layers.npz")
    R = rollouts['rollout']

    count_R, important_columns_R = column_importance_count(R, percentile=90)

    count_R_rel, important_columns_R_rel = column_importance_relative(R, alpha=ALPHA)
    
    roll_count_dict[f"{layer}layers"] = {"count": count_R,
                                         "columns": important_columns_R}

    roll_count_dict_alpha[f"{layer}layers"] = {"count": count_R_rel,
                                               "columns": important_columns_R_rel}

with open("Results/effective_ranks/layerwise_attention_counts.pkl", "wb") as f:
    pickle.dump(attn_count_dict, f)

with open("Results/effective_ranks/rollout_counts.pkl", "wb") as f:
    pickle.dump(roll_count_dict, f)
    
with open(f"Results/effective_ranks/layerwise_attention_counts{ALPHA_s}.pkl", "wb") as f:
    pickle.dump(attn_count_dict_alpha, f)

with open(f"Results/effective_ranks/rollout_counts{ALPHA_s}.pkl", "wb") as f:
    pickle.dump(roll_count_dict_alpha, f)